# Projektaufgabe: <Titel>

**Vorlesung:** Innovative Konzepte zur Programmierung von Industrierobotern  
**Dozent:** Prof. Dr.-Ing. Björn Hein  
**Gruppe:** Alisa Hummel, Loretta Jacobs, Ole Hocker  
**Aufgabe:** 03- RRT mit kantenbewusster Erweiterung  
**Abgabedatum:** 30.07.2026  
**Vortragsdatum:** 31.07.2026

## 1. Kurzfassung
In dieser Arbeit wird das Konzept des probalisitischen Bahnplanungsverfahren "Rapidly Growing Random Trees (RRT)" um einen kantenbewussten Ansatz erweitert.

Todo: Fassen Sie kurz Problem, Ansatz, wichtigste Ergebnisse und offene Punkte zusammen.

## 2. Einleitung und Zielsetzung
_Autor: Alisa Hummel_

Der Grundaufbau des RRT-Verfahren kann folgendermaßen beschrieben werden:
* Parameter:
    * n = maximale Anzahl von generierten Knoten im Graphen
	* k = nach wie vielen neu erstellten Knoten eine Verbindung zum Zielknoten getestet wird
	* eta = Schrittweite
* Initialisierung: Prüfe Start- und Zielpunkt auf Kollision, füge Startknoten zu leerem Graphen hinzu
* Schleife: solange Anzahl Knoten < n :
	1) **Sampling** neuer random, kollisionsfreier Punkt p_rand
	2) **Nächste-Nachbar-Suche**: Bestimme nächsten Knoten q aus Graph zu Punkt p_rand
	3) **Lokale Erweiterung:** Gehe von Nachbarknoten q die Schrittweite eta in Richtung gesampeltem Punkt p_rand, dadurch neuer Punkt p_n
	4) **Kollisionsprüfung** der Verbindung zwischen dem Nachbarknoten q und dem neuen Punkt p_n
        * wenn Verbindung kollisionsfrei ist, dann p_n als neuen Knoten und Kante zwischen p_n und Nachbar q in Graph einfügen
* Zieltest: Teste nach k neu erstellten Knoten, ob der Zielpunkt kollisionsfrei mit dem Graphen verbunden werden kann. Wenn ja füge Zielknoten und Kante dem Graphen hinzu und gebe kürzesten Pfad zurück.
* Wenn die Schleife endet, also nach n Knoten keine Verbindung zum Zielknoten hergestellt wurde, dann wurde kein Lösungspfad gefunden. Das bedeutet, das RRT-Verfahren findet nicht zwingend eine Lösung, auch wenn es theoretisch einen Pfad zwischen Start und Ziel geben würde. 

Ein knotenbasierter RRT Planer nutzt für die Nächste-Nachbar-Suche eines Punktes zum Graphen lediglich die existierenden Knoten des Graphen. Wir wollen zunächst einen erweiterten, kantenbewussten RRT Planer entwickeln indem auch die Kanten des Graphen in die Nächste-Nachbar-Suche einbezogen werden. Damit kann der Nächste-Nachbar eines Punktes entweder ein vorhandener Knoten oder die Projektion des Punktes auf eine Kante des Graphen sein. Weiter wollen wir testen, wie der kantenbewusste Ansatz das Ergebnis der Pfadplanung hinsichtlich Erfolgsrate, Pfadlänge und Planungszeit verändert. Dafür testen wir das Verfahren mit verschiedenen Parameterkombinationen und in unterschiedlichen Umgebungen mit einem 2-DoF-Punktroboter und mehreren n-DoF-Planarrobotern. Die Ergebnisse vergleichen wir mit dem knotenbewussten RRT Planer und einem Bidirektionalen-RRT Planer.

Unsere Erwartungen an den kantenbewussten Ansatz im Vergleich zum knotenbewussten Ansatz sind:
1. Kürzere Lösungspfade, denn neue Kanten zwischen hinzugefügten Knoten und dem Graphen sind immer so kurz wie möglich.
2. Findet wahrscheinlicher eine Lösung, weil neu hinzugefügte Kanten kürzer sind und damit eher kollisionsfrei.
3. Längere Planungszeit, weil die Nächste-Nachbar-Suche aufwendiger wird und mehr Rechenaufwand fordert. 


ToDo: Beschreiben Sie die konkrete Aufgabenstellung, eigene Teilziele und experimentelle Fragestellungen.

## 3. Ausgangscode und verwendete Module
_Autor: Loretta Jacobs_

Das Modul `IPRRT.py` enthält die Algorithmen RRTSimple und RRT, welche als Ausgangsbasis für die Implementierung der erweiterten RRT-Algorithmus dienen.

Zuerst werden in den Algorithmen Start und Ziel auf Kollision geprüft und die Dimensionen validiert (`_checkStartGoal(startList, goalList)`). Anschließend wird so lange ein zufälliger Punkt gesampelt, bis ein kollisionsfreier Punkt gefunden wird (`_getRandomFreePosition()`). Diese zwei Funktionen können für den erweiterten kantenbasierten Algorithmus ohne Änderungen übernommen werden.

Ein KDTree wird dann benutzt um den nächsten Nachbarn zu finden (`kdTree.query(pos, k=1)`). In unserem Algorithmus kann der KDTree benutzt werden um Punkte und die zugehörigen Kanten zu finden, die innerhalb eines bestimmten Radius __r__ um den Punkt __p__ liegen. Dabei muss bei `query(x, k=1, eps=0.0, p=2.0, distance_upper_bound=inf, workers=1)` die `distance_upper_bound` auf einen bestimmen Radius gesetzt werden. Diese Erweiterung wird nur implementiert, sofern sie im Rahmen des Projekts ausreichend evaluiert werden kann. Der Punkt __p__ wird dann auf die gefundenen Kanten projiziert um Lotfußpunkte für die Kanten zu finden. Mit den projezierten Punkten kann dann die nächste Nachbar Suche benutzt werden um den Lotfußpunkt zu finden der am nähesten ist.

Nachdem eine Kante gefunden wurde, die dem Graphen hinzugefügt werden soll, wird sie auf Kollision geprüft. Danach wird der neue Knoten zum Graph hinzugefügt `graph.add_node(self.lastGeneratedNodeNumber, pos=pos)` mit einer ID `lastGeneratedNodeNumber`. Die ID wird jedes mal um eins erhöht wenn ein Knoten hinzugefügt wird. Die Kante wird auch hinzugefügt `graph.add_edge(result[1], self.lastGeneratedNodeNumber)`. Dabei werden Start- und Endpunkt der Kante übergeben anhand von der ID des Knotens. Für unseren Algorithmus können wir diese Funktionen ebenfalls verwenden und vor allem sind die beim Trennen der Kante hilfreich. Beim implementieren muss auf die Nutzung von `lastGeneratedNodeNumber` für die ID's aufgepasst werden, dass es immer erhöht wird.
    
In der `Config` Dictionary (`testGoalAfterNumberOfNodes = k`) wird festgelegt, nach wie vielen angelegten Knoten k im Graphen dasZiel getestet wird (`self.lastGeneratedNodeNumber % config["testGoalAfterNumberOfNodes"]`). Das entspricht nicht zwangsläufig k Schleifendurchläufen, zum Beispiel weil eine Kollision der Kante in einem Schleifendurchlauf dazu führt, dass kein Knoten angelegt wird. Bei der Umgestaltung auf unserem Algorithmus muss auch beachtet werden, dass beim Trennen von Kanten Knoten erzeugt werden und somit in einem Schleifendurchlauf 2 Knoten hinzugefügt werden können. Daher muss entweder der Wert `k` angepasst oder eine zusätzliche Variable verwendet werden.

Die Algorithmen RRT und RRTSimple unterscheiden sich hinsichtlich der Erzeugung neuer Knoten. Während RRTSimple den gesampelten Punkt direkt als neuen Knoten verwendet und diesen mit dem nächstgelegenen Knoten verbindet, erweitert RRT den Graphen lediglich um eine Kante mit fester Schrittweite in Richtung des gesampelten Punktes (`newPos = 0.5 * (end - start) + start`). Anschließend wird dieser Punkt als neuer Knoten eingefügt und mit dem nächstgelegenen Knoten verbunden.

In den Algorithmen RRT und RRTSimple spielen Kanten bei der Auswahl des Erweiterungspunkts bisher keine Rolle, da der KDTree nur mit den Knoten im Graph arbeitet.

## 4. Konzept und Algorithmus
### Projektion Punkt auf Kante
_Autor: Alisa Hummel_

Wir berechnen den Lotfußpunkt des Punktes auf der Kante. 

* Gegeben: 
    * Kante mit Startpunkt $q_a$ und Endpunkt $q_b$
    * Punkt $p$ der projiziert werden soll
* Gesucht: 
    * projizierter Punkt $p'$
    * Parameter $t$, der die Position von $p'$ auf der Geraden durch $q_a$ und $q_b$ beschreibt
* Mathematik:
    * Vektor $AB = q_b - q_a$
    * Vektor $AP = p - q_a$
    * $t = \frac{\langle {AB,AP} \rangle}{\langle {AB,AB} \rangle}$ ; wobei $\langle {\cdot,\cdot} \rangle$ das Skalarprodukt der beiden Vektoren berechnet

Die Projektion des Punktes muss auf das Kantensegment beschränkt werden, also zwischen Start- und Endpunkt der Kante liegen ($t \in [0, 1]$). In Fällen, in denen der Lotfußpunkt außerhalb der Kante liegt, wird der projizierte Punkt der nächstgelegene Endpunkt der Kante.
* Ergebnis
    * $t<0 \Rightarrow p' = q_a$
    * $t>1 \Rightarrow p' = q_b$
    * $t \in [0, 1] \Rightarrow p' = q_a + t*q_b$

Die beschriebene Projektion ist in der Methode `PointProjection.projectPointOnEdge(p, q_a, q_b)` implementiert.


## Kante Aufteilen
_Autor: Loretta Jacobs_

Aufteilen einer Kante in zwei Teile
* Gegeben: 
    * Kante mit Startpunkt $q_{start}$ und Endpunkt $q_{end}$
    * Punkt $p$ der die Kante aufteilt
* Vorgehen:
    * Kante ($q_{start}$, $q_{end}$) vom Graph entfernen
    * Punkt $p$ zum Graph hinzufügen
    * Kanten ($q_{start}$, $p$) & ($p$, $q_{end}$) zum Graph hinzufügen

Die Funktion soll eine Exception werfen falls der Graph nicht mehr verbunden oder Kreisfrei ist. Da der entwickelte Algorithmus keine Kantengewichten verwendet, ist keine Übernahme noch eine Neuberechnung von Kantengewichten erforderlich.

Es ist keine neue Kollisionsprüfung notwendig, weil der eingefügte Punkt liegt auf der ursprünglichen Kante und somit liegen die neu erzeugten Kanten auch auf dem Verlauf der ursprünglichen Kante.

Die beschriebene Kanten Aufteilung ist in der Methode `Divide_Edge.divide_edge(self, edgePoint: Point, start_Id: int, end_Id: int)` zu finden.

## 5. Implementierung

Erläutern Sie die wichtigsten Implementierungsentscheidungen. Umfangreicher Code soll in Modulen liegen und hier importiert werden.

In [ ]:
# Eigene Module importieren
# Beispiel:
# from my_planner import MyPlanner
from BiRRT_with_Edge_sampling import BiRRTEdge
from lib.IPRRT import RRT
pass

## 6. Validierung an kleinen Beispielen
_Autor: Loretta Jacobs_

In [9]:
# Test der Punkt Projektion
from test_PointProjection import *

testPointProjection = TestPointProjection()

testPointProjection.test_projection_on_segment()
print("test_projection_on_segment passed")

testPointProjection.test_projection_outside_negative_segment()
print("test_projection_outside_negative_segment passed")

testPointProjection.test_projection_outside_positive_segment()
print("test_projection_outside_positive_segment passed")

testPointProjection.test_projection_next_negative_segment()
print("test_projection_next_negative_segment passed")

testPointProjection.test_projection_next_positive_segment()
print("test_projection_next_positive_segment passed")

test_projection_on_segment passed
test_projection_outside_negative_segment passed
test_projection_outside_positive_segment passed
test_projection_next_negative_segment passed
test_projection_next_positive_segment passed


In [10]:
# Test der Kanten Aufteilung
from test_divide_edge import TestDivideEdge

testDivideEdge = TestDivideEdge()
testDivideEdge.setUp()
testDivideEdge.test_divided_Edge()
testDivideEdge.tearDown()

print("test_divided_Edge passed")

test_divided_Edge passed


## 7. Experimente und Benchmarks

Beschreiben Sie Testumgebungen, Parameter, Metriken und Anzahl der Wiederholungen.

In [ ]:
# Benchmark-Konfigurationen definieren.
benchmarks = []
configs = []

import itertools
from typing import Any, Dict, List, Tuple, TypedDict

import numpy as np
from pandas import DataFrame

from joblib import Parallel, delayed
from lib.IPVISRRT import rrtPRMVisualize
from lib.IPPerfMonitor import IPPerfMonitor
from lib.IPBenchmark import Benchmark
import lib.IPTestSuite as ts
import matplotlib.pyplot as plt
import tqdm
import networkx as nx


class TestConfig(TypedDict):
    numberOfGeneratedNodes: int
    balanceTree: bool
    stepSize: float
    sampleGoalProbability: float
    collisionDetectionSteps: int
    maxIterations: int


NUMBER_OF_BENCHMARK_REPETITIONS = 10

COLLISION_DETECTION_STEPS = 20

MAX_ITERATIONS = 2000

sweepNumberOfGeneratedNodes = [10, 20, 50, 100, 150]
sweepStepSize = np.linspace(1, 11, 11)
sweepBalanceTree = [True] # [True, False]
sweepSampleGoalProbability = np.linspace(0.0, 1.0, 11)

numTestCases = len(sweepNumberOfGeneratedNodes) * len(sweepStepSize) * len(sweepBalanceTree) * len(sweepSampleGoalProbability)

testCases1: Dict[Tuple[int, int], Tuple[str, TestConfig]] = {}
metrics1: Dict[Tuple[int, int], Dict[str, Any]] = {}
timings1: Dict[Tuple[int, int], DataFrame] = {}


def _solution_length(solution, graph):
    """Berechne euklidische Pfadlänge einer Lösung."""
    if solution is None or solution == []:
        return np.nan
    try:
        subgraph = nx.subgraph(graph, solution)
        positions = nx.get_node_attributes(subgraph, 'pos')
        if not positions:
            return np.nan
        segment_lengths = np.linalg.norm(np.diff(list(positions.values()), axis=0), axis=1)
        return float(np.sum(segment_lengths))
    except TypeError:
        return np.nan


def _run_benchmark_case(
        benchmark: Benchmark,
        bench_rep: int,
        test_index: int,
        stepSize: float,
        balanceTree: bool,
        numGenNodes: int,
        sampleGoalProbability: float,
        collisionDetectionSteps: int,
    ) -> Dict[str, Any]:

    IPPerfMonitor.clearData()

    currentConfig = TestConfig(
        stepSize=stepSize,
        balanceTree=balanceTree,
        numberOfGeneratedNodes=numGenNodes,
        sampleGoalProbability=sampleGoalProbability,
        collisionDetectionSteps=collisionDetectionSteps,
        maxIterations=MAX_ITERATIONS
    )

    rrt = BiRRTEdge(benchmark.collisionChecker)
    solution, err = rrt.planPath(benchmark.startList, benchmark.goalList, currentConfig)
    timings = IPPerfMonitor.dataFrame()

    return {
        "key": (test_index, bench_rep),
        "benchmark_name": benchmark.name,
        "config": currentConfig,
        "solution": solution,
        "graph": rrt.graph,
        "timings": timings,
        "error": err,
    }

benchmark_tasks = []
for benchRep, benchmark in enumerate(ts.benchList * NUMBER_OF_BENCHMARK_REPETITIONS):
    for testIndex, (stepSize, balanceTree, numGenNodes, sampleGoalProbability) in enumerate(
        itertools.product(
            sweepStepSize,
            sweepBalanceTree,
            sweepNumberOfGeneratedNodes,
            sweepSampleGoalProbability,
        )
    ):
        benchmark_tasks.append(
            delayed(_run_benchmark_case)(
                benchmark,
                benchRep,
                testIndex,
                stepSize,
                balanceTree,
                numGenNodes,
                sampleGoalProbability,
                collisionDetectionSteps=COLLISION_DETECTION_STEPS
            )
        )

benchmark_results = Parallel(n_jobs=16, backend="loky")(
    tqdm.tqdm(
        benchmark_tasks,
        total=len(benchmark_tasks),
        desc="Benchmark",
    )
)

for result in benchmark_results:
    key = result["key"]
    testCases1[key] = (result["benchmark_name"], result["config"])
    metrics1[key] = {
        "solution_len": _solution_length(result["solution"], result["graph"]),
        "num_nodes": len(result["graph"].nodes()),
        "num_edges": len(result["graph"].edges()),
    }
    #timings1[key] = result["timings"]

# Hypothese: Wird die bottleneck-performance verbessert? Was ist die Wahrscheinlichkeit, dass eine Kante orthogonal zum Bottleneck liegt?
# Hypothese: Kann man die Performance verbessern, indem man projektionspunkte nicht neu erzeugt, wenn sie nicht weit von den Kantenenden entfernt sind?

In [ ]:
# plot average solution length per benchmark and config, split by sampleGoalProbability

plot_rows = []
for key, metrics in metrics1.items():
    benchmark_name, config = testCases1[key]
    plot_rows.append(
        {
            "benchmark": benchmark_name,
            "benchRep": key[1],
            "testIndex": key[0],
            "stepSize": config["stepSize"],
            "numberOfGeneratedNodes": config["numberOfGeneratedNodes"],
            "balanceTree": config["balanceTree"],
            "sampleGoalProbability": config["sampleGoalProbability"],
            "solution_len": metrics["solution_len"],
        }
    )

plot_df = DataFrame(plot_rows).copy()
success_df = plot_df.dropna(subset=["solution_len"]).copy()

if success_df.empty:
    print("No benchmark data available for plotting.")
else:
    # Wichtig: stepSize in der Groupby lassen!
    summary_df = (
        success_df.groupby(
            ["benchmark", "numberOfGeneratedNodes", "balanceTree", "sampleGoalProbability", "stepSize"],
            dropna=False,
        )
        .agg(
            mean_solution_len=("solution_len", "mean"),
            std_solution_len=("solution_len", "std"),
            runs=("solution_len", "size"),
        )
        .reset_index()
        .sort_values(["benchmark", "numberOfGeneratedNodes", "balanceTree", "sampleGoalProbability", "stepSize"])
    )

    # sampleGoalProbability Werte auswählen
    sample_goal_values = sorted(summary_df["sampleGoalProbability"].drop_duplicates().tolist())
    
    # Optional: Nur ausgewählte Werte anzeigen
    # sample_goal_values = [0.0, 0.2, 0.5, 0.8, 1.0]
    
    benchmark_names = summary_df["benchmark"].drop_duplicates().tolist()
    nrows = len(benchmark_names)
    ncols = len(sample_goal_values)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(7 * ncols, 6.8 * nrows),
        sharex=True,
        sharey=False,
    )
    axes = np.atleast_2d(axes)

    for row_index, benchmark_name in enumerate(benchmark_names):
        subset = summary_df[summary_df["benchmark"] == benchmark_name]

        for col_index, sample_goal in enumerate(sample_goal_values):
            ax = axes[row_index, col_index]
            goal_subset = subset[subset["sampleGoalProbability"] == sample_goal]
            goal_label = f"p={sample_goal:.1f}"
            table_rows = []

            for num_gen_nodes, group in goal_subset.groupby("numberOfGeneratedNodes"):
                # group enthält jetzt stepSize als Spalte
                group = group.sort_values("stepSize")
                errorbar = ax.errorbar(
                    group["stepSize"],
                    group["mean_solution_len"],
                    yerr=group["std_solution_len"],
                    marker="o",
                    linewidth=1.8,
                    capsize=3,
                    label=f"n={num_gen_nodes}",
                )
                curve_color = errorbar[0].get_color()

                min_index = group["mean_solution_len"].idxmin()
                min_row = group.loc[min_index]
                ax.scatter(
                    [min_row["stepSize"]],
                    [min_row["mean_solution_len"]],
                    marker="*",
                    s=200,
                    facecolor=curve_color,
                    edgecolor="black",
                    linewidth=1.2,
                    zorder=6,
                )
                ax.annotate(
                    f"{min_row['stepSize']:.1f}",
                    xy=(min_row["stepSize"], min_row["mean_solution_len"]),
                    xytext=(0, 10),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    color=curve_color,
                    bbox={"boxstyle": "round,pad=0.15", "facecolor": "white", "edgecolor": curve_color, "alpha": 0.85},
                )

                # Für die Tabelle: Daten aus dem originalen plot_df holen
                raw_values = plot_df[
                    (plot_df["benchmark"] == benchmark_name)
                    & (plot_df["sampleGoalProbability"] == sample_goal)
                    & (plot_df["numberOfGeneratedNodes"] == num_gen_nodes)
                ]["solution_len"]
                successful_values = raw_values.dropna()
                found_paths = int(successful_values.size)
                total_runs = int(raw_values.size)
                success_rate = found_paths / total_runs if total_runs > 0 else 0.0

                table_rows.append(
                    [
                        f"n={num_gen_nodes}",
                        f"{found_paths}/{total_runs}",
                        f"{success_rate:.2%}",
                        f"{successful_values.mean():.2f}" if found_paths > 0 else "-",
                        f"{successful_values.std():.2f}" if found_paths > 1 else "-",
                    ]
                )

            ax.set_title(f"{benchmark_name} | {goal_label}")
            ax.set_xlabel("stepSize")
            ax.set_ylabel("avg. solution path length")
            
            # Y-Achse dynamisch setzen
            max_val = summary_df["mean_solution_len"].max()
            if max_val > 0:
                ax.set_yticks(np.arange(0, max_val + 5, max(50, max_val // 10)))
            
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=8)
            ax.set_anchor("N")
            
            # X-Achse
            ax.set_xlim(summary_df["stepSize"].min() - 0.25, summary_df["stepSize"].max() + 0.25)
            x_ticks = np.arange(summary_df["stepSize"].min(), summary_df["stepSize"].max() + 0.5, 0.5)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels([f"{tick:.1f}" for tick in x_ticks])
            ax.tick_params(axis="x", which="major", labelrotation=45)
            ax.tick_params(axis="x", which="minor", length=3)

            if table_rows:
                table = ax.table(
                    cellText=table_rows,
                    colLabels=["config", "found", "rate", "avg", "std"],
                    cellLoc="center",
                    colLoc="center",
                    bbox=[0.0, -0.56, 1.0, 0.34],
                )
                table.auto_set_font_size(False)
                table.set_fontsize(7)
                table.scale(1.0, 1.05)

    fig.suptitle("Average solution length per benchmark, configuration, and sampleGoalProbability", y=1.02)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    fig

In [ ]:
# Beste Konfiguration finden (minimale durchschnittliche Lösungslänge)
best_configs = []

for benchmark_name in benchmark_names:
    for sample_goal in sample_goal_values:
        subset = summary_df[
            (summary_df["benchmark"] == benchmark_name) & 
            (summary_df["sampleGoalProbability"] == sample_goal)
        ]
        
        if not subset.empty:
            # Beste Konfiguration nach mean_solution_len
            best_row = subset.loc[subset["mean_solution_len"].idxmin()]
            
            best_configs.append({
                "benchmark": benchmark_name,
                "sampleGoalProbability": sample_goal,
                "numberOfGeneratedNodes": best_row["numberOfGeneratedNodes"],
                "stepSize": best_row["stepSize"],
                "balanceTree": best_row["balanceTree"],
                "mean_solution_len": best_row["mean_solution_len"],
                "std_solution_len": best_row["std_solution_len"],
                "runs": best_row["runs"]
            })

# Als DataFrame
best_df = DataFrame(best_configs)
print("\nBeste Konfiguration pro Benchmark und sampleGoalProbability:")
print(best_df.sort_values("runs", ascending=False).to_string(index=False))

# Test RRTEdge

In [ ]:
from RRT_with_Edge_sampling import RRTEdge, RRTEdgeConfig
from typing import TypedDict
from lib.IPPerfMonitor import IPPerfMonitor
import itertools
from typing import Any, Dict, List, Tuple, TypedDict

import numpy as np
from pandas import DataFrame

from joblib import Parallel, delayed
from lib.IPVISRRT import rrtPRMVisualize
from lib.IPPerfMonitor import IPPerfMonitor
from lib.IPBenchmark import Benchmark
import lib.IPTestSuite as ts
import matplotlib.pyplot as plt
import networkx as nx



rrtConfig = RRTEdgeConfig(
    stepSize=2,
    numberOfGeneratedNodes=500,
    sampleGoalProbability=0.1,
    collisionDetectionSteps=10,
    maxIterations=2000,
    testGoalAfterNumberOfNodes=10,
    orthogonalityMargin=0.0, # 0.0 - 0.5, 0 means strict orthogonal, 0.5 is basically RRT without edge sampling
)


for benchmark in ts.benchList:
    IPPerfMonitor.clearData()

    rrt = RRTEdge(benchmark.collisionChecker)
    solution, err = rrt.planPath(benchmark.startList, benchmark.goalList, rrtConfig)

    if err:
        print(f"Benchmark {benchmark.name} failed with error: {err}")

    fig_local = plt.figure(figsize=(10,10))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    if solution == []:
        title += " (No path found!)"
    title += "\n Assumed complexity level " + str(benchmark.level)
    
    ax.set_title(title)
    rrtPRMVisualize(rrt, solution, ax=ax, nodeSize=10)
        
    df = IPPerfMonitor.dataFrame()

# Test BiRRTEdge

In [ ]:
from BiRRT_with_Edge_sampling import BiRRTEdge, BiRRTEdgeConfig
from typing import TypedDict
from lib.IPPerfMonitor import IPPerfMonitor
import itertools
from typing import Any, Dict, List, Tuple, TypedDict

import numpy as np
from pandas import DataFrame

from lib.IPVISRRT import rrtPRMVisualize
from lib.IPPerfMonitor import IPPerfMonitor
from BenchmarkList import benchList
import matplotlib.pyplot as plt


rrtConfig = BiRRTEdgeConfig(
    stepSize=2,
    numberOfGeneratedNodes=500,
    sampleGoalProbability=0.2,
    collisionDetectionSteps=10,
    maxIterations=2000,
    orthogonalityMargin=0.1, # 0.0 - 0.5, 0 means strict orthogonal, 0.5 is basically RRT without edge sampling
    balanceTrees=True,
)


for benchmark in benchList:
    IPPerfMonitor.clearData()

    rrt = BiRRTEdge(benchmark.collisionChecker)
    solution, err = rrt.planPath(benchmark.startList, benchmark.goalList, rrtConfig)

    if err:
        print(f"Benchmark {benchmark.name} failed with error: {err}")

    fig_local = plt.figure(figsize=(10,10))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    if solution == []:
        title += " (No path found!)"
    title += "\n Assumed complexity level " + str(benchmark.level)
    
    ax.set_title(title)
    rrtPRMVisualize(rrt, solution, ax=ax, nodeSize=20)
        
    df = IPPerfMonitor.dataFrame()

In [ ]:
solutions2 = []

rrtConfig = TestConfig(
    stepSize=1,
    balanceTree=True,
    numberOfGeneratedNodes=500,
    sampleGoalProbability=0.2,
    collisionDetectionSteps=5,
    maxIterations=2000
)

IPPerfMonitor.clearData()
for benchmark in benchList:
    rrt = BiRRTEdge(benchmark.collisionChecker)
    solution, err = rrt.planPath(benchmark.startList, benchmark.goalList, rrtConfig)

    if err:
        print(f"Benchmark {benchmark.name} failed with error: {err}")

    solutions2.append(solution)

    fig_local = plt.figure(figsize=(10,10))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    if solution == []:
        title += " (No path found!)"
    title += "\n Assumed complexity level " + str(benchmark.level)
    
    ax.set_title(title)
    rrtPRMVisualize(rrt, solution, ax=ax, nodeSize=50)
        
df = IPPerfMonitor.dataFrame()

## 8. Visualisierungen und Animationen

Zeigen Sie Suchraum, Roadmap/Baum, Pfad, Kollisionen, Statistiken oder Animationen.

In [ ]:
# Visualisierungen erzeugen.
pass

## 9. Ergebnisse

Stellen Sie Ergebnisse in Tabellen und Diagrammen dar und erklären Sie beobachtete Effekte.

In [ ]:
# Ergebnisse als DataFrame/Tabelle/Plot darstellen.
pass

## 10. Diskussion

Diskutieren Sie, was funktioniert hat, wo Grenzen liegen, welche Parameter wichtig sind und wie belastbar die Ergebnisse sind.

## 11. Fazit

Fassen Sie die wichtigsten Erkenntnisse knapp zusammen.

## 12. Verwendung von KI-Werkzeugen

Dokumentieren Sie, wofür KI verwendet wurde, welche Vorschläge übernommen oder verworfen wurden und wie die Korrektheit geprüft wurde.

## 13. Präsentationsnotizen

Notieren Sie die Kernaussagen für die Präsentation: Problem, Ansatz, wichtigste Visualisierung, wichtigste Ergebnisse und wichtigste Erkenntnis.